# Week 2 — Tokenisation and the Token Economy

**ICTAII401 | ICTAII503 — Diploma of Information Technology (AI) — North Metropolitan TAFE**
📅 Week 2 · ⏱ 4 Hours · 🔷 Phase 1 — NLP Foundations

---

> **OCHRE RIDGE DISPATCH**
>
> **From:** Sandra Chen
> **Re:** The AI vendor just quoted us $4.20 per 1,000 tokens and Gus signed the contract without asking what a token is. I need you to explain token pricing to me before we commit to anything else — and tell me what we should be watching out for.

---

### What you're doing today
Last week you ran your first NLP pipeline. This week you go one level deeper — into how text gets converted into numbers before a model ever sees it. That conversion process is **tokenisation**, and it has direct cost and performance consequences for Ochre Ridge.

By the end of today you'll understand why vendors charge per token, how to measure token usage, and when a local model beats a cloud API on price.

### Learning objectives
By the end of this session you will be able to:
- Explain what a token is and how tokenisation works
- Use `tiktoken` to count and visualise tokens in any text
- Calculate estimated cloud API costs for a given workload
- Explain the trade-off between cloud and local models for high-volume use

### What to commit to GitHub this session
- `token-analysis.py` — script that tokenises 3 different texts, prints token counts, token visualisation, and cost estimate for 1,000 reports/month
- `portfolio/week-02-notes.md` — your answer to Sandra's question: *why does the vendor charge per token, and what should she watch out for?* (3–5 sentences, plain English)

---


## Setup — run this first

In [1]:
!pip install tiktoken --quiet
import tiktoken
print("tiktoken ready")

tiktoken ready


---
# Part 1 — What tokens actually are (60 min)

A model doesn't read words. It reads **tokens**.

A token is roughly a word — but not exactly. Common short words are one token each. Longer or rare words split into multiple tokens. Punctuation is usually its own token. Whitespace is folded in or stripped. The exact split depends on which tokeniser the model uses.


### Live demo — tokenise a sentence
Watch what happens to **"Hargreaves"** and **"60km/h."**

In [3]:
enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 / text-embedding-ada-002 encoding

text = "Hargreaves wants autonomous inspection robots operating at 60km/h."
tokens = enc.encode(text)

print(f"Token count: {len(tokens)}")
print(f"Tokens: {tokens}")
print(f"Decoded: {[enc.decode([t]) for t in tokens]}")

Token count: 15
Tokens: [39, 867, 265, 4798, 6944, 39293, 26767, 29807, 10565, 520, 220, 1399, 16400, 7682, 13]
Decoded: ['H', 'arg', 're', 'aves', ' wants', ' autonomous', ' inspection', ' robots', ' operating', ' at', ' ', '60', 'km', '/h', '.']


### Visualise the split

In [4]:
for t in tokens:
    print(f"{t:>7}  ->  {enc.decode([t])!r}")

     39  ->  'H'
    867  ->  'arg'
    265  ->  're'
   4798  ->  'aves'
   6944  ->  ' wants'
  39293  ->  ' autonomous'
  26767  ->  ' inspection'
  29807  ->  ' robots'
  10565  ->  ' operating'
    520  ->  ' at'
    220  ->  ' '
   1399  ->  '60'
  16400  ->  'km'
   7682  ->  '/h'
     13  ->  '.'


### 🎯 Student task — compare three encodings

Tokenise the same sentence using three different encodings and compare the counts. Same text, different splits — **this is why you can't assume token counts are portable across models.**

In [5]:
text = "Hargreaves wants autonomous inspection robots operating at 60km/h."

for encoding_name in ["cl100k_base", "p50k_base", "r50k_base"]:
    enc = tiktoken.get_encoding(encoding_name)
    tokens = enc.encode(text)
    print(f"{encoding_name}: {len(tokens)} tokens")

cl100k_base: 15 tokens
p50k_base: 15 tokens
r50k_base: 15 tokens


**Discussion:**
- Which encoding gave the highest count? The lowest?
- If Sandra gets quotes from two vendors using different models, can she compare "tokens" directly?


---
# Part 2 — Context windows (60 min)

Every model has a **context window** — the maximum number of tokens it can hold in memory at once. Input tokens and output tokens both count against it. When you exceed the window, the model forgets what came earlier.

Current context windows (approximate — check model cards for latest):
Model cards are found in huggging face https://huggingface.co/docs/hub/en/model-cards 

eg https://huggingface.co/Qwen/Qwen3-8B

| Model        | Context Window |
| ------------ | -------------- |
| GPT-4o       | 128,000 tokens |
| Llama 3.1 8B | 128,000 tokens |
| Mistral 7B   | 32,000 tokens  |
| Phi-3 medium | 128,000 tokens |


> ⚠️ **Context window ≠ practical limit**
>
> Filling a context window completely degrades model performance. A practical rule of thumb is to use no more than **80%** of the stated window. Keep this in mind for the AT1 report.

### Live demo — token count of a real maintenance report
A typical mine maintenance report is 300–500 words.

In [6]:
sample_report = """
Maintenance Report — Shaft 4 East Conveyor
Date: 2026-04-28
Technician: J. Nguyen

Inspection of conveyor belt drive assembly complete. Bearing on motor shaft
showing early signs of wear — estimated 3 weeks to failure if not replaced.
Lubrication applied to all accessible points. Belt tension within spec.
Gas sensor in Zone B reading 12 ppm methane — within safe range but trending
upward over last 14 days. Recommend daily monitoring. No other defects noted.
Next inspection due: 2026-05-05.
"""

enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(sample_report)
print(f"Report token count: {len(tokens)}")

Report token count: 119


### 🎯 Student task — how many reports fit?

Calculate how many reports you could fit in the context window of **each model** in the table above. Then apply the 80% practical-limit rule.

In [7]:
report_tokens = len(tokens)

context_windows = {
    "GPT-4o": 128_000,
    "Llama 3.1 8B": 128_000,
    "Mistral 7B": 32_000,
    "Phi-3 medium": 128_000,
}

for model, window in context_windows.items():
    practical_window = int(window * 0.8)  # 80% rule of thumb
    max_reports_theoretical = window // report_tokens
    max_reports_practical = practical_window // report_tokens
    print(f"{model:15s} theoretical: {max_reports_theoretical:5d} reports   "
          f"practical (80%): {max_reports_practical:5d} reports")

GPT-4o          theoretical:  1075 reports   practical (80%):   860 reports
Llama 3.1 8B    theoretical:  1075 reports   practical (80%):   860 reports
Mistral 7B      theoretical:   268 reports   practical (80%):   215 reports
Phi-3 medium    theoretical:  1075 reports   practical (80%):   860 reports


---
# Part 3 — Token economics (60 min)

Cloud API pricing is per token. Here's how to estimate costs for a real workload.

Assume Ochre Ridge generates **1,000 maintenance reports per month**. Average report: **400 tokens input**. Each summary response: **150 tokens output**.


### Monthly and annual cost estimate

In [9]:
reports_per_month = 1000
tokens_in_per_report = 400
tokens_out_per_report = 150

# Research current pricing for the models you plan to use and fill in the values below  
price_per_1k_input = 0.005
price_per_1k_output = 0.015

monthly_input_cost = (reports_per_month * tokens_in_per_report / 1000) * price_per_1k_input
monthly_output_cost = (reports_per_month * tokens_out_per_report / 1000) * price_per_1k_output
monthly_total = monthly_input_cost + monthly_output_cost

print(f"Monthly cloud cost estimate: USD ${monthly_total:.2f}")
print(f"Annual estimate: USD ${monthly_total * 12:.2f}")

Monthly cloud cost estimate: USD $4.25
Annual estimate: USD $51.00


Now compare: running **Llama 3.1 8B locally** on the lab machine costs nothing per token. The hardware cost is already paid.

### 🎯 Student task — find the break-even point

At what monthly report volume does the cloud cost exceed **$100/month USD**? At what volume does it exceed **$1,000/month**?

In [10]:
def monthly_cost(reports, tok_in=400, tok_out=150,
                  price_in=0.005, price_out=0.015):
    cost_in = (reports * tok_in / 1000) * price_in
    cost_out = (reports * tok_out / 1000) * price_out
    return cost_in + cost_out

# Search for the break-even report volumes
for threshold in [100, 1000]:
    reports = 0
    while monthly_cost(reports) < threshold:
        reports += 1
    print(f"Cloud cost exceeds ${threshold}/month at approximately {reports} reports/month")

Cloud cost exceeds $100/month at approximately 23530 reports/month
Cloud cost exceeds $1000/month at approximately 235295 reports/month


**Discussion:** The answer to Sandra's question about token pricing is starting to take shape. At Ochre Ridge's actual volume (1,000 reports/month), where do they land relative to these thresholds — and what does that suggest about cloud vs local?

---
# Part 4 — HuggingFace tokenizers (30 min)

The HuggingFace Tokenizers library shows how tokenisation works **inside a transformer** — not just token counts, but the full **tokenise → encode → decode** loop that runs before any model inference.

This is the same tokenisation process your Gus pipeline will use in Week 6.

### Complete the interactive exercises
- [HuggingFace LLM Course — Chapter 6](https://huggingface.co/learn/llm-course/chapter6/1) — complete sections **1, 2 and 3**
- Focus on the `encode` and `decode` methods and the `word_ids()` function
- **Do not** attempt the training-from-scratch section


### Optional live demo — HuggingFace tokenizer in this notebook

If `transformers` is installed in your environment, this shows the encode/decode loop directly (skip if offline — the HF course link above covers the same ground interactively).

In [12]:
pip install transformers

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/11.6 MB ? eta -:--:--
   --------- ------------------------------ 2.9/11.6 MB 22.1 MB/s eta 0:00:01
   ---------------------------------------- 11.6/11.6 MB 51.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 94.5 MB/s  0:00:00

   -------- ------------------------------- 2/9 [mdurl]
   ----------------- ---------------------- 4/9 [markdown-it-py]
   ----------------- ---------------------- 4/9 [markdown-it-py]
   ----------------- ---------------------- 4/9 [markdown-it-py]
   ----------------- ---------------------- 4/9 [markdown-it-py]
   ---------------------- ----------------- 5/9 [rich]
   ---------------------- ----------------- 5/9 [rich]
   ---------------------- ----------------- 5/9 [rich]
   ---------------------- ----------------- 5/9 [rich]
   ---------------

In [2]:
pip install --pre torch torchvision torchaudio --index-url https://pytorch.org

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pytorch.org
Note: you may need to restart the kernel to use updated packages.


In [9]:
from transformers import AutoTokenizer


# 1. Load a tokenizer — try "bert-base-uncased" to start
hf_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Hargreaves wants autonomous inspection robots operating at 60km/h."

# 2. Encode the text — what method turns text into input_ids?
#    (see HF course chapter 6, section 2)
encoded = hf_tok(text, return_tensors="pt")

# 3. Convert those ids back into readable token strings
#    (there's a tokenizer method for this — check convert_ids_to_tokens)
inputs_with_offsets = hf_tok(text, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()

# 4. Decode the ids all the way back to a string
decoded = hf_tok.decode(inputs_with_offsets)

print("Input IDs:", encoded)
print("Tokens:", tokens)
print("Decoded:", decoded)

# 5. Bonus — use word_ids() to see which tokens belong to which
#    original word. How does this compare to what tiktoken showed you
#    in Part 1?

Input IDs: {'input_ids': tensor([[  101,  5292, 10623, 16416,  6961,  4122,  8392, 10569, 13507,  4082,
          2012,  3438, 22287,  1013,  1044,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
Tokens: ['[CLS]', 'ha', '##rg', '##rea', '##ves', 'wants', 'autonomous', 'inspection', 'robots', 'operating', 'at', '60', '##km', '/', 'h', '.', '[SEP]']
Decoded: [CLS] hargreaves wants autonomous inspection robots operating at 60km / h. [SEP]


---
# Wrap-up — Back to Sandra

You now have everything you need to answer Sandra's original question.

> **From:** Sandra Chen
> **Re:** The AI vendor just quoted us $4.20 per 1,000 tokens and Gus signed the contract without asking what a token is. I need you to explain token pricing to me before we commit to anything else — and tell me what we should be watching out for.

**Before you leave today, draft your answer** (3–5 sentences, plain English) covering:
1. What a token actually is
2. Why token counts differ across models/vendors (Part 1)
3. Why volume matters — context windows and cost scale together (Parts 2–3)
4. What Ochre Ridge should watch out for before signing the next contract

This becomes `portfolio/week-02-notes.md`. Pair it with `token-analysis.py` (tokenise 3 texts, print counts + visualisation + the 1,000-report cost estimate) and commit both to GitHub before you go.


In [10]:
# Draft your answer to Sandra here, then move it into portfolio/week-02-notes.md

sandra_answer = """
A token is a small piece of text that an AI model processes instead of whole words, and different models may split the same text into different numbers of tokens.
Because each model uses its own tokenizer, token counts and pricing cannot be compared directly between different AI vendors.
The number of tokens affects both the model's context window (how much information it can remember at once) and the cost, since cloud AI services charge based on the number of input and output tokens processed.
Before signing another contract, Ochre Ridge should compare the vendor's tokenizer, pricing model, context window, and expected monthly token usage to ensure the service is cost-effective and suitable for their workload.
"""
print(sandra_answer)


A token is a small piece of text that an AI model processes instead of whole words, and different models may split the same text into different numbers of tokens.
Because each model uses its own tokenizer, token counts and pricing cannot be compared directly between different AI vendors.
The number of tokens affects both the model's context window (how much information it can remember at once) and the cost, since cloud AI services charge based on the number of input and output tokens processed.
Before signing another contract, Ochre Ridge should compare the vendor's tokenizer, pricing model, context window, and expected monthly token usage to ensure the service is cost-effective and suitable for their workload.



---
## Unit Mapping

| What you did                                               | Unit      | Element/PC |
| ------------------------------------------------------------ | --------- | ---------- |
| Used tiktoken to tokenise mine report text                 | ICTAII503 | PC 2.2     |
| Identified sentence and phrase boundaries via tokenisation | ICTAII503 | PC 2.3     |
| Researched model specifications and context windows        | ICTAII401 | PC 1.4     |

## Resources
- [HuggingFace LLM Course — Chapter 6](https://huggingface.co/learn/llm-course/chapter6/1) — complete sections 1–3, stop before the tokenizer training section
- [tiktoken on GitHub](https://github.com/openai/tiktoken) — install with `pip install tiktoken`, run the basic usage examples in the README

**Navigation:** ← Week 1 | [Course Overview] | Week 3 →
